# **자료구조 참고 사례**



---



## 사례: **Google 검색 엔진- Inverted Index**

Google 검색 엔진의 Inverted Index 구조는 Brin & Page (1998) 논문 [The Anatomy of a Large-Scale Hypertextual Web Search Engine](http://infolab.stanford.edu/pub/papers/google.pdf) 에서 최초로 상세히 설명되며, 이 논문이 역색인 자료구조의 근거가 됨

### **역색인**(Inverted Index)이란?
| 구분 | 설명 | 예시 |
| :--- | :--- | :--- |
| **역색인 (Inverted Index)** | 단어로부터 해당 단어가 포함된 문서 목록을 찾아내는 매핑 구조 | "blue" → [Doc1, Doc3] |
| **Stopword (불용어)** | 검색 인덱스 생성 시 의미가 없어 제외되는 일반적인 단어들 | "a", "the", "is", "it" 등 |
| **Hash Map** | 평균 $O(1)$의 검색 속도를 보장하여 역색인을 구현하는 핵심 자료구조 | Python의 `dict` (딕셔너리) |
| **Forward Index (순방향 색인)** | 문서별로 포함된 단어들을 나열한 구조 (역색인의 반대 방향) | Doc1 → ["bright", "blue", "butterfly", ...] |

### **실습 예제 1** — 기본 역색인 구현

In [ ]:
# =====================================================
# 실습 1: 역색인(Inverted Index) 기본 구현
# =====================================================

from collections import defaultdict
import re

# ── 1. 원본 문서 (이미지와 동일)
documents = {
    1: "The bright blue butterfly hangs on the breeze.",
    2: "It's best to forget the great sky and to retire from every wind.",
    3: "Under blue sky, in bright sunlight, one need not search around.",
}

# ── 2. 불용어(Stopword) 목록 (이미지와 동일)
STOPWORDS = {
    "a", "and", "around", "every", "for", "from",
    "in", "is", "it", "not", "on", "one", "the", "to", "under"
}

# ── 3. 텍스트 전처리 함수
def preprocess(text: str) -> list[str]:
    """소문자 변환 → 특수문자 제거 → 토큰화 → 불용어 제거"""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)   # 특수문자 제거
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens

# ── 4. 역색인 빌드
def build_inverted_index(docs: dict) -> dict:
    index = defaultdict(set)  # {term: {doc_id, ...}}
    for doc_id, text in docs.items():
        tokens = preprocess(text)
        for token in tokens:
            index[token].add(doc_id)
    # set → sorted list로 변환하는 대신 set을 그대로 반환
    return {term: doc_ids for term, doc_ids in sorted(index.items())}

inverted_index = build_inverted_index(documents)

# ── 5. 결과 출력 (이미지의 테이블과 비교)
print("=" * 45)
print(f"{'ID':>3}  {'Term':<12}  {'Documents'}")
print("=" * 45)
for idx, (term, doc_ids) in enumerate(inverted_index.items(), start=1):
    print(f"{idx:>3}  {term:<12}  {sorted(list(doc_ids))}") # 출력 시에는 정렬된 리스트로 변환

print("\n 전체 고유 단어 수:", len(inverted_index))


In [ ]:
# 해당하는 단어가 있는 문서 찾기
query = "blue sky".split()

# 첫 번째 쿼리 단어에 해당하는 문서 ID 집합을 초기 결과로 설정
result = set(inverted_index[query[0]])

# 나머지 쿼리 단어들에 대해 반복하면서 교집합 연산을 수행 (AND 조건)
for q in query[1:]:
    # 현재 쿼리 단어에 해당하는 문서 ID 집합과 기존 결과의 교집합을 구함
    result = result & set(inverted_index[q])

print(f"단어가 있는 문서: {result}") # 최종 검색 결과 (모든 쿼리 단어를 포함하는 문서 ID 집합) 출력

### **실습 예제 2** — Boolean 검색 엔진 구현 (AND / OR / NOT)

In [ ]:
# =====================================================
# 실습 2: Boolean 검색 엔진
# AND, OR, NOT 연산으로 문서 검색
# =====================================================

# ── 역색인 빌드
inverted_index = build_inverted_index(documents)    # dict
all_doc_ids = set(documents.keys())                 # set


# ── Boolean 검색 함수
def search(query: str) -> set:
    """
    지원 문법:
      blue AND sky       → 두 단어 모두 포함
      blue OR wind       → 둘 중 하나 포함
      NOT butterfly      → 포함하지 않는 문서
      blue AND NOT wind  → blue 포함 & wind 미포함
    """
    query = query.lower().strip()

    # NOT 처리
    if query.startswith("not "):
        term = query[4:].strip()
        return all_doc_ids - inverted_index.get(term, set())

    # AND NOT 처리
    if " and not " in query:
        left, right = query.split(" and not ")
        return inverted_index.get(left.strip(), set()) - inverted_index.get(right.strip(), set())

    # AND 처리
    if " and " in query:
        terms = [t.strip() for t in query.split(" and ")]
        result = all_doc_ids.copy()
        for t in terms:
            result &= inverted_index.get(t, set())
        return result

    # OR 처리
    if " or " in query:
        terms = [t.strip() for t in query.split(" or ")]
        result = set()
        for t in terms:
            result |= inverted_index.get(t, set())
        return result

    # 단일 단어
    return inverted_index.get(query, set())


# ── 검색 테스트
queries = [
    "blue",
    "blue AND sky",
    "blue OR wind",
    "NOT butterfly",
    "sky AND NOT wind",
    "bright AND blue",
]

print("=" * 50)
print(f"{'Query':<22}  {'결과 문서 ID':<15}  {'문서 내용 미리보기'}")
print("=" * 50)
for q in queries:
    result = search(q)
    preview = " | ".join([documents[d][:30] + "..." for d in sorted(result)]) if result else "없음"
    print(f"{q:<22}  {str(sorted(result)):<15}  {preview}")

### **실습 예제 3** — TF-IDF 가중치 역색인 + 유사도 검색 (미니 Google)

#### **TF-IDF** (Term Frequency-Inverse Document Frequency)
TF-IDF는 정보 검색과 텍스트 마이닝에서 사용하는 가중치로, 여러 문서로 이루어진 문서군에서 특정 단어가 특정 문서 내에서 얼마나 중요한지를 나타내는 통계적 수치

- **등장**: Karen Spärck Jones가 1972년 IDF(역문서빈도)의 통계적 해석을 제안했으며, 이것이 TF-IDF의 핵심 토대 가 됨
- **목적**: **문서 내 키워드 추출, 문서 간 유사도 측정, 검색 결과 순위 결정(Ranking)**
- **장점**: 구현이 쉽고, 불용어(Stopword) 처리 효과가 자연스럽게 포함됨
- **단점**: 단어의 순서나 문맥(Context)을 고려하지 못함 (Bag of Words 기반의 한계)
- **활용**: **구글 검색 엔진의 초기 랭킹 알고리즘, 뉴스 핵심 키워드 자동 추출**



#### **TF-IDF 수식**
$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$

- TF: $ f(t, d) $
    - 문서 $d$에서 단어 $t$가 나타난 횟수
- IDF: $\log \frac{N}{n_t}$
    - $N$: 전체 문서 개수 / $n_t$: 단어 $t$가 포함된 문서 개수


| 변수 | 의미 | 역할 |
| :--- | :--- | :--- |
| **$t$** | Term | 분석하고자 하는 개별 단어 |
| **$d$** | Document | 단어가 포함된 특정 문서 |
| **$D$** | Documents | 비교 대상이 되는 전체 문서 집합 |
| **$N$** | Total Number | 전체 문서의 총 개수 |
| **$n_t$** | Document Frequency | 단어 $t$가 한 번이라도 등장한 문서의 수 |

In [ ]:
# =====================================================
# 실습 3: TF-IDF 가중치 역색인 + 코사인 유사도 검색
# "진짜 검색엔진"의 핵심 원리를 구현합니다.
# =====================================================

import re
import math
from collections import defaultdict

# ── 확장된 문서 모음 (수업용)
documents = {
    1: "The bright blue butterfly hangs on the breeze.",
    2: "It's best to forget the great sky and to retire from every wind.",
    3: "Under blue sky, in bright sunlight, one need not search around.",
    4: "Blue butterflies fly under the bright morning sky.",
    5: "The wind and breeze make the sky feel great today.",
}

STOPWORDS = {
    "a","and","around","every","for","from","in","is","it",
    "not","on","one","the","to","under","make","feel","today",
    "fly","morning"
}

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return [t for t in text.split() if t not in STOPWORDS]

# ── TF 계산 (단어 빈도 / 문서 총 단어 수)
def compute_tf(tokens):
    tf = defaultdict(float)
    for t in tokens:
        tf[t] += 1
    total = len(tokens)
    return {t: count / total for t, count in tf.items()}

# ── IDF 계산 (log(전체 문서 수 / 해당 단어 포함 문서 수))
def compute_idf(docs_tokens):
    N = len(docs_tokens)
    idf = defaultdict(float)
    all_terms = set(t for tokens in docs_tokens.values() for t in tokens)
    for term in all_terms:
        df = sum(1 for tokens in docs_tokens.values() if term in tokens)
        idf[term] = math.log(N / df) + 1  # smoothing
    return idf

# ── 역색인 + TF-IDF 벡터 빌드
docs_tokens = {doc_id: preprocess(text) for doc_id, text in documents.items()}
idf = compute_idf(docs_tokens)

tfidf_vectors = {}
inverted_index = defaultdict(dict)  # {term: {doc_id: tfidf_score}}

for doc_id, tokens in docs_tokens.items():
    tf = compute_tf(tokens)
    vec = {}
    for term, tf_val in tf.items():
        score = tf_val * idf[term]
        vec[term] = score
        inverted_index[term][doc_id] = score
    tfidf_vectors[doc_id] = vec

# ── 코사인 유사도 계산
def cosine_similarity(vec1, vec2):
    common = set(vec1) & set(vec2)
    dot = sum(vec1[t] * vec2[t] for t in common)
    norm1 = math.sqrt(sum(v**2 for v in vec1.values()))
    norm2 = math.sqrt(sum(v**2 for v in vec2.values()))
    return dot / (norm1 * norm2) if norm1 and norm2 else 0.0

# ── 검색 함수
def tfidf_search(query: str, top_k: int = 3):
    query_tokens = preprocess(query)
    if not query_tokens:
        return []

    # 쿼리 후보 문서: 역색인에서 빠르게 필터링
    candidate_docs = set()
    for term in query_tokens:
        if term in inverted_index:
            candidate_docs.update(inverted_index[term].keys())

    # 쿼리 TF-IDF 벡터
    query_tf = compute_tf(query_tokens)
    query_vec = {t: query_tf[t] * idf.get(t, 0) for t in query_tokens}

    # 유사도 계산 및 정렬
    scores = []
    for doc_id in candidate_docs:
        sim = cosine_similarity(query_vec, tfidf_vectors[doc_id])
        scores.append((doc_id, sim))

    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

# ── 검색 실행
test_queries = ["blue sky", "bright butterfly", "wind breeze great"]

print("=" * 65)
for q in test_queries:
    print(f"\n 검색어: '{q}'")
    print(f"  {'순위':<4} {'Doc ID':<8} {'유사도':>8}  문서 내용")
    print("  " + "-" * 55)
    results = tfidf_search(q)
    if results:
        for rank, (doc_id, score) in enumerate(results, 1):
            print(f"  {rank:<4} Doc {doc_id:<4}   {score:.4f}   {documents[doc_id][:45]}...")
    else:
        print("  검색 결과 없음")

# ── 역색인 + TF-IDF 테이블 출력
print("\n\nTF-IDF 역색인 테이블 (상위 10개 term)")
print(f"{'Term':<12}", end="")
for doc_id in sorted(documents.keys()):
    print(f"  {'Doc'+str(doc_id):>8}", end="")
print()
print("-" * 55)
sorted_terms = sorted(inverted_index.items())[:10]
for term, doc_scores in sorted_terms:
    print(f"{term:<12}", end="")
    for doc_id in sorted(documents.keys()):
        score = doc_scores.get(doc_id, 0)
        print(f"  {score:>8.4f}", end="")
    print()

- scikit-learn에 TfidfVectorizer 가 있음
- 실제 Google은 → PageRank, BERT 임베딩으로 확장



---



## 사례: **Redis – Skip List**

### **Redis란?**

- Redis(Remote Dictionary Server)는 인메모리(In-Memory) 기반의 오픈 소스 Key-Value 구조 NoSQL 데이터베이스


| 구분 | 상세 내용 | 비유 및 특징 |
| :--- | :--- | :--- |
| **기본 정의** | **메모리 내(In-Memory)** 데이터 구조 저장소 | "전 세계에서 가장 빠른 메모리 위 전용 창고" |
| **자료구조** | String, List, Set, Hash, **HyperLogLog**, Stream 등 지원 | 단순 저장소가 아닌 '**자료구조 자체를 서비스**'하는 서버 |
| **지능적 특징** | **확률적 자료구조** 지원 (예: HyperLogLog) | 지능형 자료구조의 표본, '메모리 극 가성비' 실현 |
| **주요 용도** | 캐싱, 실시간 순위표, 세션 관리, 메시지 브로커 | "DB 앞단에서 초고속 처리를 담당하는 **가속기**" |


### Redis 자료형

| Redis 자료형 | 내부 구현 자료구조 (Implementation) | 사용 이유 |
| :--- | :--- | :--- |
| **String / Hash** | **Hash Table** | 초고속 키-값 매핑 ($O(1)$) |
| **Sorted Set** | **Hash Table + Skip List** | ID로 조회($O(1)$) + 점수 순 정렬 및 범위 검색($O(\log n)$) |
| **List** | **Quicklist (Linked List + ZipList)** | 양끝 데이터 삽입/삭제 최적화 |

### **실습 예제 1**: 간단 Skip List 구현

In [ ]:
import random

class SkipNode:
    def __init__(self, key, value, level):
        self.key = key
        self.value = value
        self.forward = [None] * (level + 1)  # 각 레벨의 다음 포인터

class SkipList:
    MAX_LEVEL = 16
    P = 0.5  # 레벨 승격 확률

    def __init__(self, min_key=None): # min_key 파라미터 추가
        if min_key is None:
            # 기본값은 숫자를 가정하여 -float('inf')
            self.header = SkipNode(-float('inf'), None, self.MAX_LEVEL)
        else:
            # min_key가 제공되면 해당 값으로 헤더 초기화 (예: '' for strings)
            self.header = SkipNode(min_key, None, self.MAX_LEVEL)
        self.level = 0  # 현재 최대 레벨

    def random_level(self):
        lvl = 0
        while random.random() < self.P and lvl < self.MAX_LEVEL:
            lvl += 1
        return lvl

    def insert(self, key, value):
        update = [None] * (self.MAX_LEVEL + 1)
        current = self.header

        for i in range(self.level, -1, -1):
            while current.forward[i] and current.forward[i].key < key:
                current = current.forward[i]
            update[i] = current

        lvl = self.random_level()
        if lvl > self.level:
            for i in range(self.level + 1, lvl + 1):
                update[i] = self.header
            self.level = lvl

        new_node = SkipNode(key, value, lvl)
        for i in range(lvl + 1):
            new_node.forward[i] = update[i].forward[i]
            update[i].forward[i] = new_node

    def search(self, key):
        current = self.header
        for i in range(self.level, -1, -1):
            while current.forward[i] and current.forward[i].key < key:
                current = current.forward[i]
        current = current.forward[0]
        if current and current.key == key:
            return current.value
        return None

    def delete(self, key):
        update = [None] * (self.MAX_LEVEL + 1)
        current = self.header
        for i in range(self.level, -1, -1):
            while current.forward[i] and current.forward[i].key < key:
                current = current.forward[i]
            update[i] = current

        target = current.forward[0]
        if target and target.key == key:
            for i in range(self.level + 1):
                if update[i].forward[i] != target:
                    break
                update[i].forward[i] = target.forward[i]

    def display(self):
        print("=== Skip List 구조 ===")
        for i in range(self.level, -1, -1):
            node = self.header.forward[i]
            row = f"Level {i}: "
            while node:
                row += f"[{node.key}] → "
                node = node.forward[i]
            print(row + "None")

# 사용 예시
sl = SkipList()
for k in [10, 20, 30, 40, 50, 70, 80]:
    sl.insert(k, f"val_{k}")

sl.display()
print("\n탐색 30:", sl.search(30))
sl.delete(30)
print("삭제 후 탐색 30:", sl.search(30))


- 랭킹 시스템 (게임 점수판)

In [ ]:
# Skip List로 실시간 랭킹 정렬
sl = SkipList()
players = [("Alice", 3500), ("Bob", 1200), ("Charlie", 4800), ("Diana", 2100)]
for name, score in players:
    sl.insert(score, name)

sl.display()  # 점수 기준 정렬된 랭킹 확인

- 사전(Dictionary) 자동완성 기반 탐색

In [ ]:
# 단어 저장 후 범위 탐색
words = SkipList(min_key='') # 문자열 키를 위한 초기화
for w in ["apple", "apricot", "banana", "cherry", "date"]:
    words.insert(w, True)

# 'a'로 시작하는 단어 탐색 (범위 탐색 확장 가능)
words.display()
word = 'banana'
print(f"\n탐색 '{word}':", words.search(word)) # words SkipList에서 'apple' 검색

### **Redis 서버 설치 및 실행**

Redis 서버는 Python 코드와 별도로 실행되어야 하는 인메모리 데이터베이스입니다. Colab 환경에서 Redis를 사용하려면 먼저 설치하고 시작해야 합니다. 아래 코드는 Colab 환경에 Redis 서버를 설치하고 백그라운드에서 실행하는 명령어입니다.

In [ ]:
!apt-get update
!apt-get install redis-server

# Redis 서버를 백그라운드에서 실행
!redis-server --daemonize yes

Redis 서버가 시작되었으니, 이제 Redis 클라이언트를 사용하여 데이터에 접근할 수 있습니다.

In [ ]:
!pip install redis

In [ ]:
# 일반적인 파이썬 딕셔너리 (내 컴퓨터 메모리)
my_dict = {"user:1": "홍길동"}

# Redis (외부 서버 메모리 - Remote Dictionary)
import redis
import time

# Redis 서버가 완전히 시작될 때까지 잠시 대기 (선택 사항이지만 안정성 향상)
time.sleep(2)

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# 딕셔너리처럼 저장 및 조회
r.set("user:1", "홍길동") # O(1)
print(r.get("user:1"))      # O(1)

### **실습 예제 1** — 실시간 게임 랭킹 시스템


- redis의 다양한 함수:https://redisgate.kr/redis/command/zsets.php

In [ ]:
import redis
import random
import time

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# ─── 설정 ───────────────────────────────────────────
LEADERBOARD_KEY = "game:leaderboard"
players = ["Alice", "Bob", "Carol", "Dave", "Eve",
           "Frank", "Grace", "Heidi", "Ivan", "Judy"]

# ─── 1. 플레이어 점수 초기화 ─────────────────────────
print("=" * 50)
print(" 게임 랭킹 시스템 — Redis Sorted Set")
print("=" * 50)

for player in players:
    score = random.randint(1000, 10000)
    r.zadd(LEADERBOARD_KEY, {player: score})
    print(f"  추가: {player:8s} → {score:,}점")

# ─── 2. 전체 Top 10 조회 (내림차순) ─────────────────
print("\n📊 TOP 10 랭킹")
print("-" * 30)
top10 = r.zrevrange(LEADERBOARD_KEY, 0, 9, withscores=True)
for rank, (player, score) in enumerate(top10, 1):
    print(f"  {rank:2d}위  {player:8s}  {int(score):,}점")

# ─── 3. 특정 플레이어 순위 조회 — O(log n) ───────────
target = "Alice"
rank = r.zrevrank(LEADERBOARD_KEY, target)  # 0-indexed
score = r.zscore(LEADERBOARD_KEY, target)   # O(1) ← Hash Table!
print(f"\n{target} 조회")
print(f"  순위: {rank + 1}위")
print(f"  점수: {int(score):,}점  ← O(1) Hash Table 조회!")

# ─── 4. 점수 갱신 후 순위 변화 관찰 ─────────────────
print(f"\n{target} 점수 +5,000 추가!")
r.zincrby(LEADERBOARD_KEY, 5000, target)
new_rank = r.zrevrank(LEADERBOARD_KEY, target)
new_score = r.zscore(LEADERBOARD_KEY, target)
print(f"  변경 후 순위: {new_rank + 1}위")
print(f"  변경 후 점수: {int(new_score):,}점")

# ─── 5. 점수 구간 조회 (8000점 이상) ─────────────────
print("\n8,000점 이상 플레이어")
high_scorers = r.zrangebyscore(LEADERBOARD_KEY, 8000, "+inf", withscores=True)
for player, score in sorted(high_scorers, key=lambda x: -x[1]):
    print(f"  {player:8s}  {int(score):,}점")

# 정리
r.delete(LEADERBOARD_KEY)


### **실습 예제 2** — Skip List vs 일반 정렬 리스트 성능 비교

In [ ]:
import redis
import time
import random
import bisect

# ─────────────────────────────────────────────────────
# Redis Sorted Set(Skip List) vs Python sorted list 비교
# ─────────────────────────────────────────────────────
print("=" * 60)
print(" Skip List(Redis) vs Sorted List 성능 비교")
print("=" * 60)

r = redis.Redis(host='localhost', port=6379, decode_responses=True)
KEY = "perf:test"
N = 10_000  # 데이터 수

# ─── 데이터 생성 ─────────────────────────────────────
data = [(f"user_{i}", random.randint(1, 1_000_000)) for i in range(N)]

# ─── 1. Redis Sorted Set 삽입 ────────────────────────
r.delete(KEY)
start = time.perf_counter()
pipe = r.pipeline()
for member, score in data:
    pipe.zadd(KEY, {member: score})
pipe.execute()
redis_insert = time.perf_counter() - start

# ─── 2. Python sorted list 삽입 ──────────────────────
sorted_list = []  # (score, member) 튜플
start = time.perf_counter()
for member, score in data:
    bisect.insort(sorted_list, (score, member))
py_insert = time.perf_counter() - start

# ─── 3. 범위 조회 (Top 100) ──────────────────────────
start = time.perf_counter()
redis_top100 = r.zrevrange(KEY, 0, 99, withscores=True) # Modified: Use zrevrange for descending order
redis_range = time.perf_counter() - start

start = time.perf_counter()
py_top100 = sorted_list[-100:][::-1]
py_range = time.perf_counter() - start

# ─── 4. 단일 score 조회 ──────────────────────────────
target = data[N // 2][0]  # 중간 사용자
start = time.perf_counter()
for _ in range(1000):
    r.zscore(KEY, target)  # O(1) Hash Table
redis_score = time.perf_counter() - start

start = time.perf_counter()
for _ in range(1000):
    # sorted list에서 member로 score 찾기 → O(n) 선형 탐색
    result = next((s for s, m in sorted_list if m == target), None)
py_score = time.perf_counter() - start

# ─── 결과 출력 ───────────────────────────────────────
print(f"\n데이터 크기: {N:,}개\n")
print(f"{'작업':<20} {'Redis Sorted Set':>18} {'Python SortedList':>18} {'배율':>8}")
print("-" * 70)
print(f"{'삽입 (N건)':<20} {redis_insert*1000:>15.1f}ms {py_insert*1000:>15.1f}ms {py_insert/redis_insert:>7.1f}x")
print(f"{'범위 조회 Top100':<20} {redis_range*1000:>15.3f}ms {py_range*1000:>15.3f}ms {'비슷':>8}")
print(f"{'score 조회 x1000':<20} {redis_score*1000:>15.1f}ms {py_score*1000:>15.1f}ms {py_score/redis_score:>7.1f}x")

print("\n# 핵심 결론:")
print("  - ZSCORE가 빠른 이유: Hash Table O(1) 조회")
print("  - 삽입이 빠른 이유: Skip List는 재정렬 불필요")
print("  - Python sorted list: member→score 역방향 조회 시 O(n) 병목!")

r.delete(KEY)
